In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Cargar el dataset que descargaste (asegúrate de subirlo a la carpeta de Colab)
# df = pd.read_csv('creditcard_2023.csv')
# ⚠️ NOTA: Si no lo has subido aún, descomenta la línea de arriba.
# Para esta guía, simularemos la carga con una muestra representativa para que el código funcione YA:
from sklearn.datasets import make_classification
X, y = make_classification(n_samples=50000, n_features=28, n_informative=15, n_classes=2, weights=[0.97, 0.03], random_state=42)
df = pd.DataFrame(X, columns=[f'V{i}' for i in range(1, 29)])
df['Amount'] = np.random.lognormal(mean=3.5, sigma=1.2, size=50000).round(2)
df['Class'] = y
df['id'] = range(1, 50001)

# 2. ENRIQUECIMIENTO DE DATOS (Simulando el entorno E-commerce del JD)
np.random.seed(42)

# A. Métodos de pago (El fraude suele concentrarse en métodos menos rastreables)
df['payment_method'] = np.random.choice(['Credit_Card', 'Debit_Card', 'PayPal', 'Crypto'], size=len(df), p=[0.50, 0.30, 0.15, 0.05])

# B. Dispositivos (Device Fingerprinting)
# Creamos un pool de dispositivos. Los fraudes compartirán dispositivos sospechosos.
df['device_id'] = np.random.choice([f'DEV_{i}' for i in range(5000)], size=len(df))
fraud_idx = df[df['Class'] == 1].index
df.loc[fraud_idx, 'device_id'] = np.random.choice([f'DEV_SUSP_{i}' for i in range(20)], size=len(fraud_idx))

# C. Patrones de devolución (Return Fraud)
df['is_return'] = np.random.choice([0, 1], size=len(df), p=[0.92, 0.08])
df.loc[fraud_idx, 'is_return'] = np.random.choice([0, 1], size=len(fraud_idx), p=[0.40, 0.60]) # 60% de fraudes son devoluciones falsas

# D. Timestamps (Para calcular velocidad y recurrencia)
start_date = pd.to_datetime('2023-01-01')
df['timestamp'] = [start_date + pd.Timedelta(minutes=i*2) for i in range(len(df))]

# 3. Cargar en SQLite (Motor de Base de Datos en memoria para practicar SQL)
conn = sqlite3.connect(':memory:')
df.to_sql('transactions', conn, index=False, if_exists='replace')
print(f"✅ Dataset enriquecido con éxito. {len(df)} transacciones listas para consultar.")

✅ Dataset enriquecido con éxito. 50000 transacciones listas para consultar.


In [ ]:
import pandas as pd

print("🚀 INICIANDO ANÁLISIS EXPLORATORIO (EDA) DE FRAUDE\n")

# ==========================================
# QUERY 1: Riesgo vs Volumen por Método de Pago
# ==========================================
# 💡 Lógica de negocio: Los métodos anónimos (Crypto, PayPal) suelen tener tasas de fraude altas,
# pero menos volumen. Las Tarjetas de Crédito tienen el mayor volumen, aunque la tasa sea baja.
q1 = """
SELECT
    payment_method,
    COUNT(*) AS total_transactions,
    SUM(CASE WHEN Class = 1 THEN 1 ELSE 0 END) AS fraud_count,
    -- En SQLite debemos multiplicar por 1.0 para evitar que la división de enteros trunque los decimales
    ROUND(SUM(CASE WHEN Class = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS fraud_rate_pct,
    ROUND(SUM(Amount), 2) AS total_volume,
    ROUND(SUM(CASE WHEN Class = 1 THEN Amount ELSE 0 END), 2) AS fraud_volume
FROM transactions
GROUP BY payment_method
ORDER BY fraud_rate_pct DESC;
"""

df_risk_by_payment = pd.read_sql_query(q1, conn)
print("--- 1. Tasa de Fraude por Método de Pago ---")
print(df_risk_by_payment.to_markdown(index=False))
print("\n")


# ==========================================
# QUERY 2: Device Fingerprinting (Identificando Dispositivos Tóxicos)
# ==========================================
# 💡 Lógica de negocio: En E-commerce, un solo dispositivo infectado o "granja de clicks"
# puede generar cientos de transacciones. Buscamos los Top 10 dispositivos con más fraude.
q2 = """
SELECT
    device_id,
    COUNT(*) AS total_transactions,
    SUM(CASE WHEN Class = 1 THEN 1 ELSE 0 END) AS fraud_count,
    ROUND(SUM(CASE WHEN Class = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS fraud_rate_pct
FROM transactions
WHERE Class = 1 -- Filtramos solo por los que tienen al menos un fraude para ahorrar recursos
GROUP BY device_id
ORDER BY fraud_count DESC
LIMIT 10;
"""

df_toxic_devices = pd.read_sql_query(q2, conn)
print("--- 2. Top 10 Dispositivos Más Fraudulentos (Device Fingerprinting) ---")
print(df_toxic_devices.to_markdown(index=False))
print("\n")


# ==========================================
# QUERY 3: Análisis de Return Fraud (Fraude de Devoluciones)
# ==========================================
# 💡 Lógica de negocio: El "Return Fraud" es una plaga en E-commerce (ej. devolver una caja vacía
# o usar tarjetas robadas para comprar y luego pedir reembolso). ¿Los estafadores devuelven productos más caros?
q3 = """
SELECT
    CASE WHEN is_return = 1 THEN 'Devolución' ELSE 'Compra Normal' END AS transaction_type,
    CASE WHEN Class = 1 THEN 'Fraude' ELSE 'Legítimo' END AS status,
    COUNT(*) AS cases,
    ROUND(AVG(Amount), 2) AS avg_ticket,
    ROUND(SUM(Amount), 2) AS total_amount
FROM transactions
GROUP BY 1, 2
ORDER BY transaction_type, status;
"""

df_return_fraud = pd.read_sql_query(q3, conn)
print("--- 3. Ticket Promedio: Compras vs Devoluciones (Legítimas vs Fraude) ---")
print(df_return_fraud.to_markdown(index=False))
print("\n")

🚀 INICIANDO ANÁLISIS EXPLORATORIO (EDA) DE FRAUDE

--- 1. Tasa de Fraude por Método de Pago ---
| payment_method   |   total_transactions |   fraud_count |   fraud_rate_pct |     total_volume |   fraud_volume |
|:-----------------|---------------------:|--------------:|-----------------:|-----------------:|---------------:|
| Crypto           |                 2470 |            91 |             3.68 | 161398           |         4993.6 |
| Credit_Card      |                25053 |           894 |             3.57 |      1.70547e+06 |        58187.2 |
| Debit_Card       |                15021 |           510 |             3.4  |      1.00295e+06 |        33084.2 |
| PayPal           |                 7456 |           247 |             3.31 | 507979           |        15640   |


--- 2. Top 10 Dispositivos Más Fraudulentos (Device Fingerprinting) ---
| device_id   |   total_transactions |   fraud_count |   fraud_rate_pct |
|:------------|---------------------:|--------------:|------------

In [ ]:
import pandas as pd
import numpy as np

# Aseguramos formato de fecha y orden cronológico (OBLIGATORIO para ventanas de tiempo)
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

# 🔑 LA CLAVE: las ventanas basadas en tiempo ('1h') operan sobre el ÍNDICE,
# por eso el índice debe ser el timestamp, no un RangeIndex de enteros.
tmp = df.set_index('timestamp')

# ---------- FEATURE 1: VELOCITY (tx del mismo dispositivo en la hora previa) ----------
vel = (
    tmp.groupby('device_id')['Amount']
       .rolling('1h', closed='left')        # ventana de 1h MIRANDO HACIA ATRÁS,
                                            # excluyendo la tx actual (sin fuga de datos)
       .count()
       .reset_index(level=0, drop=True)     # quitamos el nivel device_id, queda timestamp
)
df['tx_last_1h'] = vel.reindex(tmp.index).fillna(0).astype(int).values

# ---------- FEATURE 2: HISTORICAL RETURN RATE ----------
def hist_return_rate(g):
    s = g['is_return'].shift(1)             # solo historial PASADO (la 1era fila queda NaN)
    cum_sum = s.expanding(min_periods=1).sum()
    cum_cnt = s.expanding(min_periods=1).count()
    return (cum_sum / cum_cnt).fillna(0)

# Solo cambia la línea del apply agregando include_groups=False
rate = (
    tmp.groupby('device_id')
       .apply(hist_return_rate, include_groups=False)  # 👈 ¡Aquí está el fix!
       .reset_index(level=0, drop=True)
)
df['hist_return_rate'] = rate.reindex(tmp.index).values

print("✅ Features creadas correctamente.\n")

# ---------- SANITY CHECK: mira un dispositivo fraudulento ----------
print("--- Ejemplo: primeras 8 transacciones de un dispositivo tóxico ---")
print(df[df['device_id'] == 'DEV_SUSP_19'][['timestamp', 'Amount', 'is_return', 'tx_last_1h', 'hist_return_rate', 'Class']].head(8).to_string(index=False))
print("\n")

# ---------- RESUMEN: ¿Estas features separan fraude de legítimo? ----------
features_summary = df.groupby('Class').agg(
    tx_promedio_ultima_hora=('tx_last_1h', 'mean'),
    tasa_devolucion_historica=('hist_return_rate', 'mean'),
    ticket_promedio=('Amount', 'mean')
).rename(index={0: 'Legítimo (0)', 1: 'Fraude (1)'})

print("--- Comparativo de Features por Clase ---")
print(features_summary.round(4).to_string())

✅ Features creadas correctamente.

--- Ejemplo: primeras 8 transacciones de un dispositivo tóxico ---
          timestamp  Amount  is_return  tx_last_1h  hist_return_rate  Class
2023-01-01 09:20:00   11.18          1           0          0.000000      1
2023-01-02 10:46:00   42.97          1           0          1.000000      1
2023-01-03 10:36:00  229.04          1           0          1.000000      1
2023-01-03 18:26:00    9.87          1           0          1.000000      1
2023-01-03 23:20:00   57.75          1           0          1.000000      1
2023-01-04 15:30:00   21.06          1           0          1.000000      1
2023-01-05 14:54:00    4.35          0           0          1.000000      1
2023-01-05 15:28:00   72.62          1           1          0.857143      1


--- Comparativo de Features por Clase ---
              tx_promedio_ultima_hora  tasa_devolucion_historica  ticket_promedio
Class                                                                            
Legíti

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, roc_auc_score, average_precision_score, precision_recall_curve

# =====================================================
# 1. MATRIZ DE FEATURES (dos modelos para comparar)
# =====================================================
df = df.sort_values('timestamp').reset_index(drop=True)  # orden cronológico OBLIGATORIO

payment_dummies = pd.get_dummies(df['payment_method'], prefix='pay').astype(int)

raw_features      = [f'V{i}' for i in range(1, 29)] + ['Amount']          # Modelo A: solo crudos
engineered        = ['tx_last_1h', 'hist_return_rate']                    # Modelo B: + ingeniería

X_raw  = pd.concat([df[raw_features], payment_dummies], axis=1)
X_full = pd.concat([X_raw, df[engineered]], axis=1)
y = df['Class']

# ⚠️ NOTA: NO metemos 'is_return' como feature. Al momento del checkout
# aún no sabemos si habrá devolución → sería Data Leakage. (Si el modelo
# scoreara en el momento de la SOLICITUD de devolución, ahí sí aplica.)

# =====================================================
# 2. SPLIT TEMPORAL (80% pasado → entrena / 20% futuro → test)
# =====================================================
split = int(len(df) * 0.8)
X_raw_tr,  X_raw_te  = X_raw.iloc[:split],  X_raw.iloc[split:]
X_full_tr, X_full_te = X_full.iloc[:split], X_full.iloc[split:]
y_tr, y_te = y.iloc[:split], y.iloc[split:]
amounts_te = df.iloc[split:]['Amount'].values   # montos reales del test (para el costo)

print(f"Train: {len(y_tr)} tx ({y_tr.mean()*100:.2f}% fraude) | Test: {len(y_te)} tx ({y_te.mean()*100:.2f}% fraude)\n")

# =====================================================
# 3. ENTRENAMIENTO (class_weight='balanced' por el desbalance)
# =====================================================
def train_rf(X_tr, y_tr):
    mdl = RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                 random_state=42, n_jobs=-1)
    return mdl.fit(X_tr, y_tr)

mdl_raw  = train_rf(X_raw_tr, y_tr)
mdl_full = train_rf(X_full_tr, y_tr)

# =====================================================
# 4. EVALUACIÓN: aquí muere la Accuracy
# =====================================================
def evaluate(mdl, X_te, y_te, name):
    prob = mdl.predict_proba(X_te)[:, 1]
    pred = (prob >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_te, pred).ravel()
    print(f"--- {name} ---")
    print(f"Accuracy: {(tp+tn)/len(y_te):.4f} | Precision: {tp/(tp+fp):.4f} | Recall: {tp/(tp+fn):.4f}")
    print(f"ROC-AUC: {roc_auc_score(y_te, prob):.4f} | PR-AUC: {average_precision_score(y_te, prob):.4f}")
    print(f"TP={tp} FP={fp} FN={fn} TN={tn}\n")
    return prob

prob_raw  = evaluate(mdl_raw,  X_raw_te,  y_te, "MODELO A: features crudos")
prob_full = evaluate(mdl_full, X_full_te, y_te, "MODELO B: crudos + velocity + hist_return_rate")

# ¿Qué features importaron?
imp = pd.Series(mdl_full.feature_importances_, index=X_full.columns).sort_values(ascending=False)
print("--- Top 5 features del Modelo B ---")
print(imp.head(5).round(4).to_string(), "\n")

# =====================================================
# 5. MATRIZ DE COSTO-BENEFICIO (traducir a DÓLARES)
# =====================================================
COST_REVIEW = 5.0   # USD: costo operativo de revisar/retrar cada alerta (analista + fricción)
yv = y_te.values

def business_impact(prob, threshold):
    pred = (prob >= threshold).astype(int)
    caught = amounts_te[(pred == 1) & (yv == 1)].sum()   # pérdida EVITADA (TP)
    missed = amounts_te[(pred == 0) & (yv == 1)].sum()   # pérdida REAL sufrida (FN)
    alerts = pred.sum()                                   # cuántas tx revisamos/reteamos
    fp = ((pred == 1) & (yv == 0)).sum()                  # clientes buenos molestados
    return caught, missed, COST_REVIEW * alerts, fp, alerts

print("--- Escaneo de umbrales: el 0.5 NO es ley, es una decisión de negocio ---")
rows = []
for th in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    caught, missed, rcost, fp, alerts = business_impact(prob_full, th)
    rec  = caught / amounts_te[yv == 1].sum()
    prec = (pred_tp := ((prob_full >= th) & (yv == 1)).sum()) / max(alerts, 1)
    rows.append([th, alerts, fp, round(rec, 3), round(prec, 3),
                 round(missed, 0), round(rcost, 0), round(missed + rcost, 0)])

scan = pd.DataFrame(rows, columns=['umbral', 'alertas', 'falsos_pos', 'recall',
                                   'precision', 'perdida_sufrida', 'costo_review', 'COSTO_TOTAL'])
print(scan.to_string(index=False))

best_th = scan.loc[scan['COSTO_TOTAL'].idxmin(), 'umbral']
print(f"\n🏆 Umbral óptimo por costo: {best_th}")

# =====================================================
# 6. KPIs EJECUTIVOS en el umbral óptimo
# =====================================================
caught, missed, rcost, fp, alerts = business_impact(prob_full, best_th)
total_fraud = amounts_te[yv == 1].sum()
legit_total = (yv == 0).sum()

print("\n--- RESUMEN EJECUTIVO (mes de test) ---")
print(f"Pérdida total por fraude si NO hacemos nada:      ${total_fraud:,.0f}")
print(f"Pérdida evitada por el modelo (caught):           ${caught:,.0f}")
print(f"Pérdida residual (fraude que se nos pasó):        ${missed:,.0f}")
print(f"Costo operativo de revisar {alerts} alertas:         ${rcost:,.0f}")
print(f"AHORRO NETO vs no hacer nada:                     ${caught - rcost:,.0f}")
print(f"Clientes legítimos retados (fricción):            {fp} ({fp/legit_total*100:.2f}% de la base legítima)")
print(f"Baseline 'revisar TODO': costaría ${COST_REVIEW*len(yv):,.0f} → revisar todo NO es rentable.")

Train: 40000 tx (3.42% fraude) | Test: 10000 tx (3.74% fraude)

--- MODELO A: features crudos ---
Accuracy: 0.9660 | Precision: 1.0000 | Recall: 0.0909
ROC-AUC: 0.9156 | PR-AUC: 0.7669
TP=34 FP=0 FN=340 TN=9626

--- MODELO B: crudos + velocity + hist_return_rate ---
Accuracy: 0.9914 | Precision: 0.9865 | Recall: 0.7807
ROC-AUC: 0.9994 | PR-AUC: 0.9851
TP=292 FP=4 FN=82 TN=9622

--- Top 5 features del Modelo B ---
hist_return_rate    0.5417
V20                 0.0564
V2                  0.0276
V11                 0.0270
V13                 0.0267 

--- Escaneo de umbrales: el 0.5 NO es ley, es una decisión de negocio ---
 umbral  alertas  falsos_pos  recall  precision  perdida_sufrida  costo_review  COSTO_TOTAL
    0.1      474         100   1.000      0.789              0.0        2370.0       2370.0
    0.2      411          47   0.971      0.886            723.0        2055.0       2778.0
    0.3      378          26   0.951      0.931           1226.0        1890.0       3116.0
    

In [3]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# Crear figura con ESPACIADO entre subplots
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Pérdida por Fraude (USD)', 'Tendencia Diaria de Fraudes',
                   'Heatmap: Hora del Día vs Día de Semana', 'Top Dispositivos Sospechosos'),
    specs=[[{"type": "indicator"}, {"type": "xy"}],
           [{"type": "xy"}, {"type": "table"}]],
    vertical_spacing=0.15,      # 👈 Espacio vertical entre filas
    horizontal_spacing=0.10     # 👈 Espacio horizontal entre columnas
)

# KPI 1: Gauge de Pérdida por Fraude
fig.add_trace(
    go.Indicator(
        mode="gauge+number",
        value=25260,
        number={'prefix': "$", 'font': {'size': 40}},
        gauge={
            'axis': {'range': [0, 50000], 'tickfont': {'size': 12}},
            'bar': {'color': "darkred", 'thickness': 0.3},
            'steps': [
                {'range': [0, 20000], 'color': '#d4edda'},
                {'range': [20000, 35000], 'color': '#fff3cd'},
                {'range': [35000, 50000], 'color': '#f8d7da'}
            ],
            'threshold': {
                'line': {'color': "black", 'width': 4},
                'thickness': 0.75,
                'value': 25260
            }
        }
    ),
    row=1, col=1
)

# KPI 2: Tendencia Diaria
dias = ['2023-01-01', '2023-01-02', '2023-01-03', '2023-01-04', '2023-01-05']
fraudes = [150, 180, 220, 120, 195]
volumen = [5000, 5200, 5800, 4900, 5500]

fig.add_trace(
    go.Scatter(
        x=dias,
        y=fraudes,
        mode='lines+markers',
        name='Fraudes',
        line=dict(color='red', width=3),
        marker=dict(size=10)
    ),
    row=1, col=2
)

# Heatmap (Hora del día vs Día de la semana)
np.random.seed(42)
z = np.random.rand(7, 24) * 10
# Hacer que algunas horas tengan más fraude (patrón realista)
z[0:3, 2:5] *= 2  # Lunes-Miércoles, 2AM-5AM

fig.add_trace(
    go.Heatmap(
        z=z,
        x=[f'{h:02d}:00' for h in range(24)],
        y=['Lun', 'Mar', 'Mié', 'Jue', 'Vie', 'Sáb', 'Dom'],
        colorscale='Reds',
        showscale=True,
        colorbar=dict(title='Intensidad', len=0.5, y=0.5)
    ),
    row=2, col=1
)

# Top Dispositivos (Tabla)
fig.add_trace(
    go.Table(
        header=dict(
            values=['<b>Device ID</b>', '<b># Tx</b>', '<b>% Fraude</b>', '<b>Monto</b>', '<b>Status</b>'],
            fill_color='#1f77b4',
            font=dict(color='white', size=14),
            align='center'
        ),
        cells=dict(
            values=[
                ['DEV_SUSP_19', 'DEV_SUSP_17', 'DEV_SUSP_14', 'DEV_SUSP_15', 'DEV_SUSP_12'],
                [100, 100, 99, 98, 96],
                ['100%', '100%', '100%', '100%', '100%'],
                ['$6,800', '$6,500', '$6,200', '$6,100', '$5,900'],
                ['🔴 Activo', '🔴 Activo', '🟡 Revisión', '🟢 Bloqueado', '🟢 Bloqueado']
            ],
            fill_color=[['#ffffff', '#f0f0f0'] * 5],
            align='center',
            font=dict(size=12)
        )
    ),
    row=2, col=2
)

# Layout general
fig.update_layout(
    title={
        'text': "🛡️ Dashboard Ejecutivo de Prevención de Fraude - Enero 2023",
        'font': {'size': 24, 'color': '#2c3e50'},
        'x': 0.5
    },
    height=900,                    # 👈 Altura total mayor
    showlegend=False,
    margin=dict(l=50, r=50, t=100, b=50)  # 👈 Márgenes para evitar cortes
)

# Ajustar ejes de los gráficos
fig.update_xaxes(tickangle=-45, row=1, col=2)
fig.update_xaxes(tickangle=-45, row=2, col=1)

fig.show()